<a href="https://colab.research.google.com/github/Pranshu244/Financial-Fraud-Detection/blob/main/Model_Training/Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch_geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
from datetime import timedelta
from collections import defaultdict
import bisect
import torch
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.nn import GraphSAGE
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix

In [ ]:
df1=pd.read_csv("transactions.csv")
df2=pd.read_csv("accounts.csv")

In [ ]:
df1.head()

,transaction_id,timestamp,source_account,destination_account,amount,channel,transaction_type,is_suspicious,pattern_id,pattern_type
0,TXN000001,2026-06-01 02:21:53,ACC00571,ACC00563,4197.52,CASH_DEPOSIT,cash_deposit,0,NaN,normal
1,TXN000002,2026-06-01 03:34:36,ACC00245,ACC00407,14933.53,UPI,transfer,0,NaN,normal
2,TXN000003,2026-06-01 03:48:04,ACC00093,ACC00096,7274.49,UPI,transfer,0,NaN,normal
3,TXN000004,2026-06-01 04:00:34,ACC00473,ACC00445,5472.44,UPI,transfer,0,NaN,normal
4,TXN000005,2026-06-01 04:51:05,ACC00624,ACC00488,4377.72,IMPS,transfer,0,NaN,normal


In [ ]:
df2.head()

,account_id,account_type,opened_days_ago,region,baseline_monthly_txn_count,baseline_avg_amount,dormancy_flag,true_label,all_roles
0,ACC00001,individual,913,North,15.2,14272.94,0,structuring_source,structuring_source
1,ACC00002,individual,60,North,19.9,8410.23,0,mule,mule
2,ACC00003,individual,1061,East,9.7,7185.30,0,mule,mule
3,ACC00004,individual,70,South,13.2,1287.82,0,mule,mule
4,ACC00005,individual,113,South,6.9,6124.01,0,mule,mule


In [ ]:
df1.shape

(7622, 10)

In [ ]:
df2.shape

(700, 9)

In [ ]:
df1.isnull().sum()

,0
transaction_id,0
timestamp,0
source_account,0
destination_account,0
amount,0
channel,0
transaction_type,0
is_suspicious,0
pattern_id,7000
pattern_type,0


In [ ]:
df1.duplicated().sum()

np.int64(0)

In [ ]:
df2.isnull().sum()

,0
account_id,0
account_type,0
opened_days_ago,0
region,0
baseline_monthly_txn_count,0
baseline_avg_amount,0
dormancy_flag,0
true_label,0
all_roles,0


In [ ]:
df2.duplicated().sum()

np.int64(0)

In [ ]:
df1.describe()

,amount,is_suspicious
count,7.622000e+03,7622.000000
mean,1.323900e+04,0.081606
std,6.488776e+04,0.273781
min,1.152400e+02,0.000000
25%,2.075012e+03,0.000000
50%,3.928685e+03,0.000000
75%,7.847740e+03,0.000000
max,1.071882e+06,1.000000


In [ ]:
df2.describe()

,opened_days_ago,baseline_monthly_txn_count,baseline_avg_amount,dormancy_flag
count,700.000000,700.000000,700.000000,700.000000
mean,307.114286,12.022143,6533.148729,0.017143
std,279.892553,4.758420,5330.367479,0.129896
min,10.000000,1.000000,627.490000,0.000000
25%,98.500000,8.900000,3165.957500,0.000000
50%,222.500000,12.100000,5086.020000,0.000000
75%,436.250000,15.200000,8070.435000,0.000000
max,1740.000000,24.900000,72904.970000,1.000000


In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7622 entries, 0 to 7621
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       7622 non-null   object 
 1   timestamp            7622 non-null   object 
 2   source_account       7622 non-null   object 
 3   destination_account  7622 non-null   object 
 4   amount               7622 non-null   float64
 5   channel              7622 non-null   object 
 6   transaction_type     7622 non-null   object 
 7   is_suspicious        7622 non-null   int64  
 8   pattern_id           622 non-null    object 
 9   pattern_type         7622 non-null   object 
dtypes: float64(1), int64(1), object(8)
memory usage: 595.6+ KB


In [ ]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   account_id                  700 non-null    object 
 1   account_type                700 non-null    object 
 2   opened_days_ago             700 non-null    int64  
 3   region                      700 non-null    object 
 4   baseline_monthly_txn_count  700 non-null    float64
 5   baseline_avg_amount         700 non-null    float64
 6   dormancy_flag               700 non-null    int64  
 7   true_label                  700 non-null    object 
 8   all_roles                   700 non-null    object 
dtypes: float64(2), int64(2), object(5)
memory usage: 49.3+ KB


In [ ]:
df1["timestamp"] = pd.to_datetime(df1["timestamp"])
print(df1["timestamp"].dtype)
print("Start:", df1["timestamp"].min())
print("End:", df1["timestamp"].max())

datetime64[ns]
Start: 2026-06-01 02:21:53
End: 2026-07-16 22:40:45


In [ ]:
accounts = set(df2["account_id"])
missing_sources = set(df1["source_account"]) - accounts
missing_destinations = set(df1["destination_account"]) - accounts
print("Missing source accounts:", len(missing_sources))
print("Missing destination accounts:", len(missing_destinations))

Missing source accounts: 0
Missing destination accounts: 0


In [ ]:
print(df1["timestamp"].is_monotonic_increasing)

True


In [ ]:
account_to_node = {account_id: idx for idx, account_id in enumerate(df2["account_id"])}
print("Number of nodes:", len(account_to_node))
print("ACC00001 →", account_to_node["ACC00001"])

Number of nodes: 700
ACC00001 → 0


In [ ]:
G = nx.MultiDiGraph()

for _, row in df2.iterrows():
    G.add_node(
        account_to_node[row["account_id"]],
        account_id=row["account_id"],
        account_type=row["account_type"],
        opened_days_ago=row["opened_days_ago"],
        region=row["region"],
        baseline_monthly_txn_count=row["baseline_monthly_txn_count"],
        baseline_avg_amount=row["baseline_avg_amount"],
        dormancy_flag=row["dormancy_flag"]
    )
for _, row in df1.iterrows():
    source = account_to_node[row["source_account"]]
    destination = account_to_node[row["destination_account"]]

    G.add_edge(
        source,
        destination,
        transaction_id=row["transaction_id"],
        timestamp=row["timestamp"],
        amount=row["amount"],
        channel=row["channel"],
        transaction_type=row["transaction_type"]
    )
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 700
Edges: 7622


In [ ]:
u, v, key, data = next(iter(G.edges(keys=True, data=True)))
print("Source node:", u)
print("Destination node:", v)
print("Edge key:", key)
print("Edge data:", data)

Source node: 0
Destination node: 217
Edge key: 0
Edge data: {'transaction_id': 'TXN000333', 'timestamp': Timestamp('2026-06-03 09:52:34'), 'amount': 1317.03, 'channel': 'RTGS', 'transaction_type': 'transfer'}


In [ ]:
structuring = df1[df1["pattern_type"] == "structuring"]
print("Structuring transactions:", len(structuring))
print("\nUnique pattern IDs:")
print(structuring["pattern_id"].nunique())
print("\nTransactions per pattern:")
print(structuring.groupby("pattern_id").size().describe())

Structuring transactions: 354

Unique pattern IDs:
15

Transactions per pattern:
count    15.000000
mean     23.600000
std       5.110214
min      15.000000
25%      21.000000
50%      25.000000
75%      28.000000
max      30.000000
dtype: float64


In [ ]:
print(structuring[["pattern_id", "timestamp", "source_account","destination_account", "amount"]].head(20))

    pattern_id           timestamp source_account destination_account  \
7    STRUCT011 2026-06-01 05:19:05       ACC00634            ACC00340   
10   STRUCT011 2026-06-01 06:18:33       ACC00634            ACC00008   
13   STRUCT011 2026-06-01 07:53:56       ACC00634            ACC00466   
16   STRUCT011 2026-06-01 09:31:00       ACC00634            ACC00468   
25   STRUCT011 2026-06-01 10:50:09       ACC00634            ACC00448   
28   STRUCT011 2026-06-01 11:20:04       ACC00634            ACC00564   
35   STRUCT011 2026-06-01 12:31:58       ACC00634            ACC00485   
40   STRUCT011 2026-06-01 13:44:03       ACC00634            ACC00116   
46   STRUCT011 2026-06-01 16:06:10       ACC00634            ACC00056   
52   STRUCT011 2026-06-01 16:51:03       ACC00634            ACC00365   
63   STRUCT011 2026-06-01 17:36:13       ACC00634            ACC00059   
65   STRUCT011 2026-06-01 17:49:55       ACC00634            ACC00248   
85   STRUCT011 2026-06-01 21:34:26       ACC00634  

In [ ]:
structuring_span = (
    structuring
    .groupby("pattern_id")["timestamp"]
    .agg(["min", "max"])
)
structuring_span["duration_hours"] = (
    structuring_span["max"] - structuring_span["min"]
).dt.total_seconds() / 3600
print(structuring_span)
print("\nDuration statistics:")
print(structuring_span["duration_hours"].describe())

                           min                 max  duration_hours
pattern_id                                                        
STRUCT001  2026-07-11 04:40:56 2026-07-12 12:10:04       31.485556
STRUCT002  2026-06-12 12:02:39 2026-06-13 19:39:32       31.614722
STRUCT003  2026-06-12 03:21:15 2026-06-13 13:44:06       34.380833
STRUCT004  2026-07-11 20:02:44 2026-07-13 01:07:16       29.075556
STRUCT005  2026-06-16 02:00:33 2026-06-17 11:58:05       33.958889
STRUCT006  2026-06-02 18:28:30 2026-06-04 03:00:55       32.540278
STRUCT007  2026-07-11 06:45:54 2026-07-12 16:48:00       34.035000
STRUCT008  2026-07-05 12:52:57 2026-07-06 21:22:47       32.497222
STRUCT009  2026-06-14 18:33:27 2026-06-16 02:30:06       31.944167
STRUCT010  2026-06-08 20:24:34 2026-06-10 05:34:28       33.165000
STRUCT011  2026-06-01 05:19:05 2026-06-02 13:44:54       32.430278
STRUCT012  2026-07-05 10:33:21 2026-07-06 22:08:07       35.579444
STRUCT013  2026-07-12 19:38:06 2026-07-14 01:46:21       30.13

In [ ]:
destination_stats = (
    structuring
    .groupby("pattern_id")["destination_account"]
    .nunique()
)
print(destination_stats)
print("\nDestination statistics:")
print(destination_stats.describe())

pattern_id
STRUCT001    15
STRUCT002    28
STRUCT003    26
STRUCT004    15
STRUCT005    29
STRUCT006    24
STRUCT007    30
STRUCT008    20
STRUCT009    16
STRUCT010    26
STRUCT011    25
STRUCT012    22
STRUCT013    28
STRUCT014    28
STRUCT015    22
Name: destination_account, dtype: int64

Destination statistics:
count    15.000000
mean     23.600000
std       5.110214
min      15.000000
25%      21.000000
50%      25.000000
75%      28.000000
max      30.000000
Name: destination_account, dtype: float64


In [ ]:
source_stats = (
    structuring
    .groupby("pattern_id")["source_account"]
    .nunique()
)

print(source_stats)

pattern_id
STRUCT001    1
STRUCT002    1
STRUCT003    1
STRUCT004    1
STRUCT005    1
STRUCT006    1
STRUCT007    1
STRUCT008    1
STRUCT009    1
STRUCT010    1
STRUCT011    1
STRUCT012    1
STRUCT013    1
STRUCT014    1
STRUCT015    1
Name: source_account, dtype: int64


In [ ]:
outgoing = df1[["timestamp", "source_account", "destination_account", "amount"]].copy()
outgoing = outgoing.sort_values("timestamp")

In [ ]:
WINDOW = timedelta(hours=24)
MIN_DESTINATIONS = 10
candidates = []
for account, group in outgoing.groupby("source_account"):
    group = group.sort_values("timestamp").reset_index(drop=True)
    left = 0
    for right in range(len(group)):
        while (
            group.loc[right, "timestamp"]
            - group.loc[left, "timestamp"]
            > WINDOW
        ):
            left += 1
        window = group.iloc[left:right + 1]
        distinct_destinations = window["destination_account"].nunique()
        if distinct_destinations >= MIN_DESTINATIONS:
            candidates.append({
                "account_id": account,
                "window_start": window["timestamp"].iloc[0],
                "window_end": window["timestamp"].iloc[-1],
                "transaction_count": len(window),
                "distinct_destinations": distinct_destinations,
                "total_amount": window["amount"].sum()
            })
            break

In [ ]:
fanout_candidates = pd.DataFrame(candidates)
print("Candidate accounts:", len(fanout_candidates))
print(fanout_candidates.head())
print(fanout_candidates.shape)

Candidate accounts: 16
  account_id        window_start          window_end  transaction_count  \
0   ACC00001 2026-07-11 20:02:44 2026-07-12 15:38:59                 11   
1   ACC00077 2026-07-12 08:37:32 2026-07-13 04:46:42                 10   
2   ACC00161 2026-07-05 10:33:21 2026-07-06 07:13:45                 10   
3   ACC00188 2026-06-02 18:28:30 2026-06-03 09:28:42                 10   
4   ACC00194 2026-07-07 12:44:30 2026-07-08 12:27:10                 10   

   distinct_destinations  total_amount  
0                     10     514747.56  
1                     10     442523.94  
2                     10     447084.71  
3                     10     495000.00  
4                     10     106160.89  
(16, 6)


In [ ]:
true_structuring_sources = set(df2.loc[df2["true_label"] == "structuring_source","account_id"])
detected_sources = set(fanout_candidates["account_id"])
true_positives = detected_sources & true_structuring_sources
false_positives = detected_sources - true_structuring_sources
missed = true_structuring_sources - detected_sources

print("True structuring sources:", len(true_structuring_sources))
print("Detected candidates:", len(detected_sources))
print("True positives:", len(true_positives))
print("False positives:", len(false_positives))
print("Missed:", len(missed))

True structuring sources: 15
Detected candidates: 16
True positives: 15
False positives: 1
Missed: 0


In [ ]:
precision = (len(true_positives) / len(detected_sources)if detected_sources else 0)
recall = len(true_positives) / len(true_structuring_sources)
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))

Precision: 0.938
Recall: 1.0


In [ ]:
layering = df1[df1["pattern_type"] == "cyclic_layering"].copy()

print("Layering transactions:", len(layering))
print("Unique pattern IDs:", layering["pattern_id"].nunique())

print("\nTransactions per pattern:")
print(
    layering
    .groupby("pattern_id")
    .size()
    .describe()
)

Layering transactions: 66
Unique pattern IDs: 12

Transactions per pattern:
count    12.000000
mean      5.500000
std       1.445998
min       3.000000
25%       4.000000
50%       6.000000
75%       7.000000
max       7.000000
dtype: float64


In [ ]:
print(
    layering[
        [
            "pattern_id",
            "timestamp",
            "source_account",
            "destination_account",
            "amount"
        ]
    ].sort_values(["pattern_id", "timestamp"])
    .head(30)
)

     pattern_id           timestamp source_account destination_account  \
644    LAYER001 2026-06-05 09:49:38       ACC00585            ACC00539   
1053   LAYER001 2026-06-07 16:32:40       ACC00539            ACC00079   
1521   LAYER001 2026-06-10 10:46:41       ACC00079            ACC00053   
1605   LAYER001 2026-06-10 23:49:11       ACC00053            ACC00453   
3258   LAYER002 2026-06-20 16:20:15       ACC00069            ACC00393   
3604   LAYER002 2026-06-22 18:49:44       ACC00393            ACC00527   
3943   LAYER002 2026-06-25 01:03:47       ACC00527            ACC00043   
4102   LAYER002 2026-06-26 05:32:20       ACC00043            ACC00622   
4186   LAYER002 2026-06-26 15:58:59       ACC00622            ACC00346   
4678   LAYER002 2026-06-29 08:00:54       ACC00346            ACC00447   
4995   LAYER002 2026-07-01 08:42:13       ACC00447            ACC00069   
3823   LAYER003 2026-06-24 05:34:40       ACC00485            ACC00562   
3911   LAYER003 2026-06-24 18:21:08   

In [ ]:
layering_span = (
    layering
    .groupby("pattern_id")["timestamp"]
    .agg(["min", "max"])
)

layering_span["duration_hours"] = (
    layering_span["max"] - layering_span["min"]
).dt.total_seconds() / 3600

print(layering_span)
print("\nDuration statistics:")
print(layering_span["duration_hours"].describe())

                           min                 max  duration_hours
pattern_id                                                        
LAYER001   2026-06-05 09:49:38 2026-06-10 23:49:11      133.992500
LAYER002   2026-06-20 16:20:15 2026-07-01 08:42:13      256.366111
LAYER003   2026-06-24 05:34:40 2026-07-02 09:18:14      195.726111
LAYER004   2026-07-05 05:00:52 2026-07-15 10:58:50      245.966111
LAYER005   2026-06-23 15:04:55 2026-06-27 17:07:53       98.049444
LAYER006   2026-06-28 16:51:15 2026-07-05 18:00:54      169.160833
LAYER007   2026-07-06 03:14:56 2026-07-13 22:56:48      187.697778
LAYER008   2026-06-15 09:34:55 2026-06-22 11:11:38      169.611944
LAYER009   2026-06-05 08:21:30 2026-06-16 22:16:49      277.921944
LAYER010   2026-06-20 12:48:15 2026-06-24 21:37:59      104.828889
LAYER011   2026-06-24 21:17:42 2026-06-29 01:55:21      100.627500
LAYER012   2026-06-23 05:05:56 2026-07-01 11:18:24      198.207778

Duration statistics:
count     12.000000
mean     178.179745


In [ ]:
for pid, group in layering.groupby("pattern_id"):
    print(f"\n--- {pid} ---")

    for _, row in group.sort_values("timestamp").iterrows():
        print(
            row["source_account"],
            "→",
            row["destination_account"],
            "|",
            row["timestamp"]
        )


--- LAYER001 ---
ACC00585 → ACC00539 | 2026-06-05 09:49:38
ACC00539 → ACC00079 | 2026-06-07 16:32:40
ACC00079 → ACC00053 | 2026-06-10 10:46:41
ACC00053 → ACC00453 | 2026-06-10 23:49:11

--- LAYER002 ---
ACC00069 → ACC00393 | 2026-06-20 16:20:15
ACC00393 → ACC00527 | 2026-06-22 18:49:44
ACC00527 → ACC00043 | 2026-06-25 01:03:47
ACC00043 → ACC00622 | 2026-06-26 05:32:20
ACC00622 → ACC00346 | 2026-06-26 15:58:59
ACC00346 → ACC00447 | 2026-06-29 08:00:54
ACC00447 → ACC00069 | 2026-07-01 08:42:13

--- LAYER003 ---
ACC00485 → ACC00562 | 2026-06-24 05:34:40
ACC00562 → ACC00175 | 2026-06-24 18:21:08
ACC00175 → ACC00580 | 2026-06-26 14:49:25
ACC00580 → ACC00061 | 2026-06-27 04:16:30
ACC00061 → ACC00338 | 2026-06-29 11:50:47
ACC00338 → ACC00485 | 2026-07-02 09:18:14

--- LAYER004 ---
ACC00592 → ACC00596 | 2026-07-05 05:00:52
ACC00596 → ACC00195 | 2026-07-06 06:56:40
ACC00195 → ACC00595 | 2026-07-08 05:49:01
ACC00595 → ACC00124 | 2026-07-10 10:08:19
ACC00124 → ACC00292 | 2026-07-12 01:20:58
ACC0

In [ ]:
MIN_HOP_AMOUNT = 200000
MIN_HOPS = 3
MAX_GAP_HOURS = 100
MAX_TOTAL_DURATION_HOURS = 300
MAX_BRANCH = 5

In [ ]:
edges = df1[["timestamp", "source_account", "destination_account", "amount"]].copy()
edges = edges[edges["amount"] >= MIN_HOP_AMOUNT].sort_values("timestamp").reset_index(drop=True)
print("Edges after amount floor:", len(edges))

outgoing_by_account = defaultdict(list)
for _, row in edges.iterrows():
    outgoing_by_account[row["source_account"]].append(
        (row["timestamp"], row["destination_account"], row["amount"])
    )
for a in outgoing_by_account:
    outgoing_by_account[a].sort(key=lambda x: x[0])

Edges after amount floor: 66


In [ ]:
def next_hops(account, current_time):
    cands = outgoing_by_account.get(account, [])
    times = [c[0] for c in cands]
    idx = bisect.bisect_right(times, current_time)
    out = []
    for ts, dst, amt in cands[idx:]:
        gap_h = (ts - current_time).total_seconds() / 3600
        if gap_h > MAX_GAP_HOURS:
            break
        out.append((ts, dst, amt))
        if len(out) >= MAX_BRANCH:
            break
    return out

In [ ]:
all_paths = []

def dfs(path, visited, start_time):
    last_ts, _, last_dst, _ = path[-1]
    if len(path) >= MIN_HOPS:
        all_paths.append(list(path))
    total_h = (last_ts - start_time).total_seconds() / 3600
    if total_h >= MAX_TOTAL_DURATION_HOURS or len(path) >= 10:
        return
    for ts, dst, amt in next_hops(last_dst, last_ts):
        is_cycle_close = (dst == path[0][1])
        if dst in visited and not is_cycle_close:
            continue
        visited.add(dst)
        path.append((ts, last_dst, dst, amt))
        dfs(path, visited, start_time)
        path.pop()
        if not is_cycle_close:
            visited.discard(dst)

for _, row in edges.iterrows():
    start = (row["timestamp"], row["source_account"], row["destination_account"], row["amount"])
    dfs([start], {row["source_account"], row["destination_account"]}, row["timestamp"])

print("Candidate chains/cycles found:", len(all_paths))

Candidate chains/cycles found: 106


In [ ]:
flagged_accounts = set()
for p in all_paths:
    flagged_accounts.add(p[0][1])
    for e in p:
        flagged_accounts.add(e[2])
print("Flagged accounts:", len(flagged_accounts))

Flagged accounts: 66


In [ ]:
true_layering = set(df2[df2["all_roles"].str.contains("layering_node", na=False)].account_id)
tp = flagged_accounts & true_layering
precision = len(tp) / len(flagged_accounts)
recall = len(tp) / len(true_layering)
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("False positives:", flagged_accounts - true_layering)
print("False negatives:", true_layering - flagged_accounts)

Precision: 1.0
Recall: 1.0
False positives: set()
False negatives: set()


In [ ]:
dorm = df1[df1["pattern_type"] == "dormant_reactivation"].copy()
print("Dormant-reactivation transactions:", len(dorm))
print("Unique pattern IDs:", dorm["pattern_id"].nunique())
print(dorm.groupby("pattern_id").size().describe())

span = dorm.groupby("pattern_id")["timestamp"].agg(["min", "max"])
span["duration_hours"] = (span["max"] - span["min"]).dt.total_seconds() / 3600
print(span["duration_hours"].describe())

Dormant-reactivation transactions: 202
Unique pattern IDs: 12
count    12.000000
mean     16.833333
std       2.855086
min      13.000000
25%      15.000000
50%      16.000000
75%      18.500000
max      22.000000
dtype: float64
count    12.000000
mean     27.377593
std       1.259484
min      25.094167
25%      26.747292
50%      27.530000
75%      28.190764
max      29.086667
Name: duration_hours, dtype: float64


In [ ]:
events = pd.concat([
    df1[["timestamp","source_account"]].rename(columns={"source_account":"account_id"}),
    df1[["timestamp","destination_account"]].rename(columns={"destination_account":"account_id"})
]).sort_values("timestamp").reset_index(drop=True)

true_dormant = set(df2[df2.dormancy_flag == 1].account_id)
normal_ids = set(df2.account_id) - true_dormant
WINDOW = timedelta(hours=24)

max_counts = []
for account, group in events.groupby("account_id"):
    if account not in normal_ids:
        continue
    group = group.sort_values("timestamp").reset_index(drop=True)
    left, best = 0, 0
    for right in range(len(group)):
        while group.loc[right,"timestamp"] - group.loc[left,"timestamp"] > WINDOW:
            left += 1
        best = max(best, right - left + 1)
    max_counts.append(best)

print(pd.Series(max_counts).describe(percentiles=[.5,.9,.95,.99]))

count    688.000000
mean       3.209302
std        2.528976
min        2.000000
50%        3.000000
90%        4.000000
95%        4.000000
99%       19.260000
max       25.000000
dtype: float64


In [ ]:
WINDOW_HOURS = 24
MIN_EVENTS = 13
K = 3

baseline_map = dict(zip(df2.account_id, df2.baseline_monthly_txn_count))
WINDOW = timedelta(hours=WINDOW_HOURS)

candidates = []
for account, group in events.groupby("account_id"):
    group = group.sort_values("timestamp").reset_index(drop=True)
    expected = max((baseline_map[account] / 30) * (WINDOW_HOURS / 24), 0.5)
    left = 0
    for right in range(len(group)):
        while group.loc[right,"timestamp"] - group.loc[left,"timestamp"] > WINDOW:
            left += 1
        count = right - left + 1
        if count >= MIN_EVENTS and count >= K * expected:
            candidates.append({"account_id": account, "peak_count_in_window": count})
            break

velocity_candidates = pd.DataFrame(candidates)
print("Candidate accounts:", len(velocity_candidates))

Candidate accounts: 25


In [ ]:
detected = set(velocity_candidates.account_id)

# strict: dormancy only
tp_dorm = detected & true_dormant
print("Recall vs dormancy_flag:", round(len(tp_dorm)/len(true_dormant), 3))
print("Precision vs dormancy_flag:", round(len(tp_dorm)/len(detected), 3))

# real precision: did it flag anything with NO fraud role at all?
any_fraud = set(df2[df2.true_label != "normal"].account_id) | true_dormant
false_alarms = detected - any_fraud
print("Accounts flagged with zero fraud role:", len(false_alarms), false_alarms)

Recall vs dormancy_flag: 1.0
Precision vs dormancy_flag: 0.48
Accounts flagged with zero fraud role: 0 set()


In [ ]:
# Cell: node features + labels
account_to_node = {a: i for i, a in enumerate(df2.account_id)}
node_df = df2.copy()
node_df["node_id"] = node_df.account_id.map(account_to_node)
node_df = node_df.sort_values("node_id").reset_index(drop=True)

acc_type_dummies = pd.get_dummies(node_df.account_type, prefix="type")
numeric = node_df[["opened_days_ago","baseline_monthly_txn_count","baseline_avg_amount","dormancy_flag"]].copy()
numeric = (numeric - numeric.mean()) / numeric.std()
X = pd.concat([numeric, acc_type_dummies], axis=1).astype(float)
x = torch.tensor(X.values, dtype=torch.float)

label_map = {"normal":0, "mule":1, "layering_node":2, "structuring_source":3}
y = torch.tensor(node_df.true_label.map(label_map).values, dtype=torch.long)

In [ ]:
# Cell: edges
src = df1.source_account.map(account_to_node).values
dst = df1.destination_account.map(account_to_node).values
edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)

In [ ]:
# Cell: ring-aware train/test split (no ring leaks across the split)
np.random.seed(42)
struct_ids = sorted(df1[df1.pattern_type=="structuring"].pattern_id.unique())
layer_ids  = sorted(df1[df1.pattern_type=="cyclic_layering"].pattern_id.unique())

def holdout_accounts(pattern_ids, frac=0.25):
    n_test = max(1, int(len(pattern_ids)*frac))
    test_ids = set(np.random.choice(pattern_ids, n_test, replace=False))
    sub = df1[df1.pattern_id.isin(test_ids)]
    return set(sub.source_account) | set(sub.destination_account)

test_fraud_accounts = holdout_accounts(struct_ids) | holdout_accounts(layer_ids)
normal_accounts = set(node_df[node_df.true_label=="normal"].account_id)
test_normal_accounts = set(np.random.choice(list(normal_accounts), int(len(normal_accounts)*0.25), replace=False))
test_accounts = test_fraud_accounts | test_normal_accounts
test_mask = torch.tensor(node_df.account_id.isin(test_accounts).values)
train_mask = ~test_mask

In [ ]:
out_deg = df1.groupby("source_account").size().reindex(df2.account_id, fill_value=0)
in_deg = df1.groupby("destination_account").size().reindex(df2.account_id, fill_value=0)
out_uniq = df1.groupby("source_account")["destination_account"].nunique().reindex(df2.account_id, fill_value=0)
in_uniq = df1.groupby("destination_account")["source_account"].nunique().reindex(df2.account_id, fill_value=0)

both = pd.concat([
    df1[["source_account","amount"]].rename(columns={"source_account":"account_id"}),
    df1[["destination_account","amount"]].rename(columns={"destination_account":"account_id"})
])
max_amt = both.groupby("account_id")["amount"].max().reindex(df2.account_id, fill_value=0)
total_amt = both.groupby("account_id")["amount"].sum().reindex(df2.account_id, fill_value=0)

node_df["out_degree"] = node_df.account_id.map(out_deg)
node_df["in_degree"] = node_df.account_id.map(in_deg)
node_df["out_unique_dest"] = node_df.account_id.map(out_uniq)
node_df["in_unique_src"] = node_df.account_id.map(in_uniq)
node_df["max_amount"] = node_df.account_id.map(max_amt)
node_df["total_amount"] = node_df.account_id.map(total_amt)

In [ ]:
node_df["flag_fanout"] = node_df.account_id.isin(set(fanout_candidates.account_id)).astype(int)
node_df["flag_layering"] = node_df.account_id.isin(flagged_accounts).astype(int)
node_df["flag_velocity"] = node_df.account_id.isin(set(velocity_candidates.account_id)).astype(int)

In [ ]:
acc_type_dummies = pd.get_dummies(node_df.account_type, prefix="type")
numeric_cols = ["opened_days_ago","baseline_monthly_txn_count","baseline_avg_amount","dormancy_flag",
                 "out_degree","in_degree","out_unique_dest","in_unique_src","max_amount","total_amount"]
numeric = node_df[numeric_cols].copy()
numeric = (numeric - numeric.mean()) / numeric.std()
flags = node_df[["flag_fanout","flag_layering","flag_velocity"]].astype(float)

X = pd.concat([numeric, acc_type_dummies, flags], axis=1).astype(float)
x = torch.tensor(X.values, dtype=torch.float)
print("New feature matrix shape:", x.shape)

data = Data(x=x, edge_index=edge_index, y=y, train_mask=train_mask, test_mask=test_mask)

New feature matrix shape: torch.Size([700, 16])


In [ ]:
class FraudGraphSAGE(torch.nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hid_dim)
        self.conv2 = SAGEConv(hid_dim, out_dim)
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        return self.conv2(x, edge_index)

torch.manual_seed(0)
model = FraudGraphSAGE(data.num_node_features, 32, 4)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
class_counts = torch.bincount(data.y[data.train_mask])
weights = 1.0 / class_counts.float()
weights = weights / weights.sum()

model.train()
for epoch in range(1, 201):
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], weight=weights)
    loss.backward()
    optimizer.step()
    if epoch % 40 == 0:
        model.eval()
        with torch.no_grad():
            pred = model(data.x, data.edge_index).argmax(dim=1)
            acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
        print(f"epoch {epoch} loss {loss.item():.4f} test_acc {acc:.4f}")
        model.train()

epoch 40 loss 0.1715 test_acc 0.8554
epoch 80 loss 0.1211 test_acc 0.8614
epoch 120 loss 0.1011 test_acc 0.8795
epoch 160 loss 0.0773 test_acc 0.9036
epoch 200 loss 0.0792 test_acc 0.8976


In [ ]:
model.eval()
with torch.no_grad():
    pred = model(data.x, data.edge_index).argmax(dim=1)

from sklearn.metrics import classification_report
print(classification_report(data.y[data.test_mask].numpy(), pred[data.test_mask].numpy(),
      target_names=["normal","mule","layering_node","structuring_source"], zero_division=0))

                    precision    recall  f1-score   support

            normal       0.96      0.86      0.91        92
              mule       0.78      0.92      0.84        50
     layering_node       1.00      1.00      1.00        20
structuring_source       0.80      1.00      0.89         4

          accuracy                           0.90       166
         macro avg       0.89      0.94      0.91       166
      weighted avg       0.91      0.90      0.90       166



In [ ]:
inv_label_map = {v: k for k, v in label_map.items()}

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index)
    probs = F.softmax(logits, dim=1)
    pred = logits.argmax(dim=1)

results = node_df.loc[test_mask.numpy(), ["account_id","true_label","all_roles"]].copy()
results["predicted_label"] = [inv_label_map[p] for p in pred[test_mask].numpy()]
results["correct"] = results["true_label"] == results["predicted_label"]
for i, cls in inv_label_map.items():
    results[f"prob_{cls}"] = probs[test_mask][:, i].numpy().round(3)

print(results.head(10).to_string(index=False))

account_id true_label all_roles predicted_label  correct  prob_normal  prob_mule  prob_layering_node  prob_structuring_source
  ACC00003       mule      mule            mule     True        0.261      0.729               0.009                    0.001
  ACC00010     normal    normal          normal     True        0.917      0.082               0.000                    0.000
  ACC00011       mule      mule            mule     True        0.339      0.657               0.002                    0.002
  ACC00015     normal    normal          normal     True        1.000      0.000               0.000                    0.000
  ACC00019       mule      mule          normal    False        0.628      0.372               0.000                    0.000
  ACC00020     normal    normal          normal     True        0.989      0.011               0.001                    0.000
  ACC00023       mule      mule            mule     True        0.250      0.749               0.001                  

In [ ]:
labels_order = ["normal","mule","layering_node","structuring_source"]
cm = confusion_matrix(results.true_label, results.predicted_label, labels=labels_order)
print(pd.DataFrame(cm, index=[f"true_{l}" for l in labels_order], columns=[f"pred_{l}" for l in labels_order]))

                         pred_normal  pred_mule  pred_layering_node  \
true_normal                       79         13                   0   
true_mule                          3         46                   0   
true_layering_node                 0          0                  20   
true_structuring_source            0          0                   0   

                         pred_structuring_source  
true_normal                                    0  
true_mule                                      1  
true_layering_node                             0  
true_structuring_source                        4  


In [ ]:
print(results[~results.correct].to_string(index=False))

account_id true_label all_roles    predicted_label  correct  prob_normal  prob_mule  prob_layering_node  prob_structuring_source
  ACC00019       mule      mule             normal    False        0.628      0.372               0.000                    0.000
  ACC00034       mule      mule             normal    False        0.859      0.136               0.002                    0.003
  ACC00063       mule      mule structuring_source    False        0.429      0.103               0.010                    0.458
  ACC00100       mule      mule             normal    False        0.730      0.268               0.001                    0.001
  ACC00165     normal    normal               mule    False        0.009      0.991               0.000                    0.000
  ACC00228     normal    normal               mule    False        0.219      0.780               0.000                    0.000
  ACC00234     normal    normal               mule    False        0.029      0.970              

In [ ]:
model.eval()
with torch.no_grad():
    all_logits = model(data.x, data.edge_index)
    all_probs = F.softmax(all_logits, dim=1)
    all_pred = all_logits.argmax(dim=1)

stage2_results = node_df[["account_id","true_label","all_roles"]].copy()
stage2_results["predicted_label"] = [inv_label_map[p] for p in all_pred.numpy()]
for i, cls in inv_label_map.items():
    stage2_results[f"prob_{cls}"] = all_probs[:, i].numpy().round(4)
stage2_results["fraud_prob"] = (1 - stage2_results["prob_normal"]).round(4)

stage2_results.to_csv("stage2_results.csv", index=False)
torch.save(model.state_dict(), "fraud_gnn_model.pt")
print(stage2_results.shape)
stage2_results.head()

(700, 9)


,account_id,true_label,all_roles,predicted_label,prob_normal,prob_mule,prob_layering_node,prob_structuring_source,fraud_prob
0,ACC00001,structuring_source,structuring_source,structuring_source,0.0000,0.0000,0.0000,1.0000,1.0000
1,ACC00002,mule,mule,mule,0.1327,0.8574,0.0095,0.0003,0.8673
2,ACC00003,mule,mule,mule,0.2609,0.7295,0.0089,0.0007,0.7391
3,ACC00004,mule,mule,mule,0.0517,0.9473,0.0008,0.0002,0.9483
4,ACC00005,mule,mule,mule,0.0753,0.9208,0.0034,0.0005,0.9247


In [ ]:
from google.colab import files
files.download("stage2_results.csv")
files.download("fraud_gnn_model.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
events = pd.concat([
    df1[["timestamp","source_account"]].rename(columns={"source_account":"account_id"}),
    df1[["timestamp","destination_account"]].rename(columns={"destination_account":"account_id"})
]).sort_values("timestamp").reset_index(drop=True)

baseline_map = dict(zip(df2.account_id, df2.baseline_monthly_txn_count))
WINDOW_HOURS = 24
WINDOW = timedelta(hours=WINDOW_HOURS)

peak_counts = {}
for account, group in events.groupby("account_id"):
    group = group.sort_values("timestamp").reset_index(drop=True)
    left, best = 0, 0
    for right in range(len(group)):
        while group.loc[right,"timestamp"] - group.loc[left,"timestamp"] > WINDOW:
            left += 1
        best = max(best, right - left + 1)
    peak_counts[account] = best

dev = pd.DataFrame({"account_id": list(peak_counts.keys()), "peak_24h_count": list(peak_counts.values())})
dev["baseline_monthly_txn_count"] = dev.account_id.map(baseline_map)
dev["expected_24h_count"] = (dev["baseline_monthly_txn_count"] / 30).clip(lower=0.5)
dev["velocity_ratio"] = dev["peak_24h_count"] / dev["expected_24h_count"]

In [ ]:
both = pd.concat([df1[["source_account","amount"]].rename(columns={"source_account":"account_id"}),
                   df1[["destination_account","amount"]].rename(columns={"destination_account":"account_id"})])
max_amt = both.groupby("account_id")["amount"].max().reindex(df2.account_id, fill_value=0)
baseline_amt_map = dict(zip(df2.account_id, df2.baseline_avg_amount))
dev["max_amount"] = dev.account_id.map(max_amt)
dev["baseline_avg_amount"] = dev.account_id.map(baseline_amt_map)
dev["amount_ratio"] = dev["max_amount"] / dev["baseline_avg_amount"].clip(lower=1)

def norm(series, cap):
    return series.clip(upper=cap) / cap

dev["velocity_score"] = norm(dev["velocity_ratio"], cap=20)
dev["amount_score"] = norm(dev["amount_ratio"], cap=15)
dev["deviation_score"] = (0.6*dev["velocity_score"] + 0.4*dev["amount_score"]).clip(upper=1.0)

In [ ]:
stage2 = pd.read_csv("stage2_results.csv")

In [ ]:
fused = stage2.merge(dev[["account_id","velocity_ratio","amount_ratio","deviation_score"]], on="account_id")
fused["final_risk"] = (0.6*fused["fraud_prob"] + 0.4*fused["deviation_score"]).round(4)

def tier(r):
    if r < 0.4: return "auto_monitor"
    elif r < 0.75: return "escalate_analyst"
    else: return "freeze_review"
fused["action_tier"] = fused["final_risk"].apply(tier)

print(pd.crosstab(fused.true_label, fused.action_tier))
fused.to_csv("stage3_final_risk.csv", index=False)

action_tier         auto_monitor  escalate_analyst  freeze_review
true_label                                                       
layering_node                  0                 0             63
mule                           3               145            104
normal                       333                37              0
structuring_source             0                 0             15
